In [28]:
!apt-get install graphviz -y
!pip install pydot

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import make_scorer
import warnings
warnings.filterwarnings('ignore')

plt.style.use('fivethirtyeight')


features = pd.read_csv("/content/drive/MyDrive/widsdatathon2023/train_data_pca_reduced.csv")
df_test = pd.read_csv('/content/drive/MyDrive/widsdatathon2023/test_data_pca.csv')

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
graphviz is already the newest version (2.42.2-6ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def fix_date_format(x):
    try:

        if x.count('/') == 2 and len(x.split('/')[0]) == 4:
            dt = pd.to_datetime(x, format='%Y/%m/%d')
        else:
            dt = pd.to_datetime(x, format='%m/%d/%y')
        return dt
    except:
        return pd.NaT

features['startdate'] = features['startdate'].apply(fix_date_format)
# Break down into three new columns: Day, Month, and Year
features['day'] = features['startdate'].dt.day
features['month'] = features['startdate'].dt.month
features['year'] = features['startdate'].dt.year
#Remove `startdate` to exclude `TimeStamp`
#features = features.drop(['startdate', 'index'], axis=1)
features = features.drop(['startdate'], axis=1)
# result
print(features[['day', 'month', 'year']].head(5))

print('Data dimensions：', features.shape)
print(features.describe())

   day  month  year
0    1      9  2014
1    2      9  2014
2    3      9  2014
3    4      9  2014
4    5      9  2014
数据维度： (375734, 53)
                PC1           PC2           PC3           PC4           PC5  \
count  3.757340e+05  3.757340e+05  3.757340e+05  3.757340e+05  3.757340e+05   
mean  -4.599105e-16 -2.904698e-17 -1.307114e-16  4.841163e-17  9.137694e-17   
std    7.561137e+00  5.286261e+00  3.983544e+00  3.499211e+00  3.123671e+00   
min   -1.678424e+01 -8.456575e+00 -1.537858e+01 -1.125110e+01 -6.275160e+00   
25%   -6.520259e+00 -4.070881e+00 -3.371606e+00 -2.548511e+00 -2.276445e+00   
50%   -1.219771e-01 -1.127291e+00  1.071022e+00  4.411403e-01 -2.254955e-02   
75%    6.700370e+00  3.201190e+00  3.014033e+00  2.381501e+00  1.899408e+00   
max    1.690323e+01  2.881767e+01  8.319471e+00  8.803222e+00  7.784658e+00   

                PC6           PC7           PC8           PC9          PC10  \
count  3.757340e+05  3.757340e+05  3.757340e+05  3.757340e+05  3.75734

In [ ]:
# Create a list of creation dates for a time series chart
years = features['year']
months = features['month']
days = features['day']
dates = [str(int(year)) + '-' + str(int(month)) + '-' + str(int(day))
         for year, month, day in zip(years, months, days)]
dates = [datetime.datetime.strptime(date, '%Y-%m-%d') for date in dates]

In [ ]:
print(features.head(5))

         PC1       PC2       PC3       PC4       PC5       PC6       PC7  \
0  12.141910 -0.059308 -1.949099 -3.691256 -5.342317 -1.582707 -3.468166   
1  12.050890 -0.148246 -2.227420 -3.858339 -5.025439 -1.472561 -3.326589   
2  11.979123 -0.257065 -2.559255 -3.947894 -4.773425 -1.415406 -3.081124   
3  11.903912 -0.365143 -2.904385 -3.935543 -4.596430 -1.357709 -2.797233   
4  11.838356 -0.558687 -3.261902 -3.897726 -4.579274 -1.315109 -2.615038   

        PC8       PC9      PC10  ...      PC44      PC45      PC46      PC47  \
0  0.435821  0.631864  0.340092  ... -1.898507  0.058186  0.443599 -1.059025   
1  1.336887  0.640830  0.519788  ... -1.930182  0.084634  0.541740 -1.025007   
2  2.170406  0.677580  0.470874  ... -1.936321  0.058884  0.588656 -0.919810   
3  2.796870  0.891793  0.200060  ... -1.862730  0.030392  0.592856 -0.852217   
4  3.204292  1.255449  0.004836  ... -1.698894  0.008344  0.308336 -0.725715   

       PC48  contest-tmp2m-14d__tmp2m  climateregions__climate

In [32]:
pc_cols = [f'PC{i}' for i in range(1, 49)]
X = features[pc_cols].values
label = features['contest-tmp2m-14d__tmp2m'].values



from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, label, test_size=0.2, random_state=42)
X_test_data = df_test[pc_cols].values

In [33]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_test_data_scaled = scaler.transform(X_test_data)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
# linear
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_test_scaled)
rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))
r2_lr = r2_score(y_test, pred_lr)
print(f"LinearRegression - RMSE: {rmse_lr:.8f}, R²: {r2_lr:.8f}")


LinearRegression - RMSE: 2.07704934, R²: 0.95576555


In [ ]:
# Ridge
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 50))
ridge_cv.fit(X_train_scaled, y_train)
pred_ridge = ridge_cv.predict(X_test_scaled)
rmse_ridge = np.sqrt(mean_squared_error(y_test, pred_ridge))
r2_ridge = r2_score(y_test, pred_ridge)
print(f"Ridge (alpha={ridge_cv.alpha_:.4f}) - RMSE: {rmse_ridge:.8f}, R²: {r2_ridge:.8f}")


Ridge (alpha=2.0236) - RMSE: 2.07704966, R²: 0.95576554


In [ ]:

# Lasso
lasso_cv = LassoCV(alphas=np.logspace(-3, 3, 30), max_iter=10000, cv=3, n_jobs=-1)
lasso_cv.fit(X_train_scaled, y_train)
pred_lasso = lasso_cv.predict(X_test_scaled)
rmse_lasso = np.sqrt(mean_squared_error(y_test, pred_lasso))
r2_lasso = r2_score(y_test, pred_lasso)
print(f"Lasso (alpha={lasso_cv.alpha_:.4f}) - RMSE: {rmse_lasso:.8f}, R²: {r2_lasso:.8f}")

Lasso (alpha=0.0010) - RMSE: 2.07701818, R²: 0.95576688


In [38]:
test_data = pd.read_csv('/content/drive/MyDrive/widsdatathon2023/test_data.csv')
model = LassoCV(alphas=np.logspace(-3, 3, 50))
model.fit(X_train_scaled, y_train)
pred = model.predict(X_test_data_scaled)

# submission file
submission = pd.DataFrame({
    'contest-tmp2m-14d__tmp2m': pred,
    'index': test_data['index']

})
submission.to_csv('submission.csv', index=False)
print("Prediction complete. The submitted file has been saved as submission.csv")

预测完成，提交文件已保存为 submission.csv
